In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BronzeToSilver").getOrCreate()

In [0]:
bronze_structured_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "bronze/air_quality_structured/"
)

structured_df = spark.read.parquet(bronze_structured_path)

structured_df.printSchema()

root
 |-- source: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- measurement_timestamp: string (nullable = true)
 |-- european_aqi: integer (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)
 |-- ingestion_timestamp_utc: string (nullable = true)



In [0]:
structured_df = structured_df.select(
    F.col("source"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("city_id"),
    F.col("city"),
    F.col("country"),
    F.to_timestamp(F.col("measurement_timestamp")).alias("measurement_timestamp"),
    F.col("european_aqi"),
    F.col("pm10"),
    F.col("pm2_5"),
    F.col("nitrogen_dioxide"),
    F.to_timestamp(F.col("ingestion_timestamp_utc")).alias("ingestion_timestamp_utc")
)

In [0]:
# Validate key values

filtered_df = structured_df.filter(
    (F.col("source") == "openmeteo") &
    (F.col("city_id").isNotNull()) &
    (F.col("measurement_timestamp").isNotNull()) &
    (F.col("european_aqi").isNotNull())
)

# Flag data

flagged_df = filtered_df.withColumns({
    "is_valid_aqi": (F.col("european_aqi").isNotNull()) & (F.col("european_aqi") >= 0),
    "is_valid_pm10": (F.col("pm10").isNotNull()) & (F.col("pm10") >= 0),
    "is_valid_pm2_5": (F.col("pm2_5").isNotNull()) & (F.col("pm2_5") >= 0),
    "is_valid_nitrogen_dioxide": (F.col("nitrogen_dioxide").isNotNull()) & (F.col("nitrogen_dioxide") >= 0),
    "is_not_future_measurement": F.col("measurement_timestamp") <= F.current_timestamp()
}
)

silver_df = flagged_df.filter(
    F.col("is_valid_aqi") &
    F.col("is_valid_pm10") &
    F.col("is_valid_pm2_5") &
    F.col("is_valid_nitrogen_dioxide") &
    F.col("is_not_future_measurement")
)

# Duduplication

dedupe_window = Window.partitionBy(
    "city_id",
    "measurement_timestamp"    
    ).orderBy(
        F.col("ingestion_timestamp_utc").desc()
    )

silver_df = silver_df.withColumn(
    "rn",
    F.row_number().over(dedupe_window)
).filter(
    F.col("rn") == 1
).drop(
    "rn"
)

In [0]:
print("Structured rows:", structured_df.count())
print("Filtered rows:", filtered_df.count())
print("Valid silver rows:", silver_df.count())

Structured rows: 720
Filtered rows: 720
Valid silver rows: 90


In [0]:
silver_path = (
    "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net/"
    "silver/air_quality/"
)

silver_df.write.mode("overwrite").parquet(silver_path)

In [0]:
spark.read.parquet(silver_path).show(20, truncate=False)
spark.read.parquet(silver_path).printSchema()

+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|source   |latitude|longitude|city_id|city        |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc   |is_valid_aqi|is_valid_pm10|is_valid_pm2_5|is_valid_nitrogen_dioxide|is_not_future_measurement|
+---------+--------+---------+-------+------------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens      |Greece |2026-07-09 00:00:00  |30          |22.7|15.3 |42.1            |2026-07-09 14:31:34.265575|true        |true         |true          |true                     |true                     |
|openmeteo|37.9838 |23.7